# Daily EnKF corrections across seven epidemic phases in Mexico

This notebook reproduces the continuous run in [first_estimation.ipynb](first_estimation.ipynb) and retains the ensemble mean immediately before and after each daily analysis. Each phase has one figure with all four compartments. A CSV contains the daily filtered rates for the entire run, from February 28, 2020 to May 9, 2023.

In [1]:
from functools import partial
from pathlib import Path
import sys

from html import escape
import json

from IPython.display import display
import numpy as np
import pandas as pd

root = Path.cwd()
if root.name == "Codigos":
    root = root.parent
sys.path.insert(0, str(root / "Codigos"))
from codes import (
    analyze_sird_ensemble, gaussian_rate_walk, get_Y_obs,
    initialize_sird_ensemble, predict_sird_ensemble,
)

DATA_DIR = root / "Datos"
SOURCE_FILE = DATA_DIR / "raw_data" / "COVID19MEXICO2020_2021_2022_2023.csv"
PHASE_FILE = DATA_DIR / "metadata" / "covid_waves_parameters.csv"
START_DATE = pd.Timestamp("2020-02-28")
LAST_DAY = 1166
END_DATE = START_DATE + pd.Timedelta(days=LAST_DAY)
CHUNK_SIZE = 250_000
POPULATION = 126_014_024
RECOVERY_DAYS = 14
SEED = 2026
ENSEMBLE_SIZE = 160
STATE_STD = np.array([1.0, 0.0, 0.0])  # People, ordered as I, R, D.
INITIAL_RATES = np.array([0.3508, 0.0714, 0.0015])  # Daily rate medians.
INITIAL_LOG_STD = np.array([0.35, 0.40, 0.50])
LOG_STEP_STD = np.array([0.12, 0.12, 0.15])  # Log-rate SD per square root of a day.
OBS_STD = np.array([10.0, 4.0, 1.5])  # Observation SDs in people, ordered as I, R, D.
OBS_COV = np.diag(OBS_STD**2)

## Calendar and observations

The phase boundaries come from the project metadata. The ensemble continues through April 2022, between waves, and through the 39 days following the sixth wave. The consolidated admission records supply cumulative positive cases and recorded deaths. Recoveries are imputed 14 days after admission for cases without a recorded death. Active cases equal cumulative positives minus recoveries and deaths.

In [2]:
phases = pd.read_csv(PHASE_FILE, parse_dates=["Start Date", "End Date"])
dates = pd.date_range(START_DATE, END_DATE, freq="D", name="Date")
phase = pd.Series("Between waves", index=dates, name="Period")
for name, start, end in phases[["Epidemiological Phase", "Start Date", "End Date"]].itertuples(index=False, name=None):
    phase.loc[start:end] = name
phase.loc[dates > phases["End Date"].max()] = "After sixth wave"
day = np.arange(len(dates))
calendar = pd.DataFrame({"Day": day, "Period": phase}, index=dates)

In [3]:
columns = ["FECHA_INGRESO", "FECHA_DEF", "CLASIFICACION_FINAL"]
observed = pd.DataFrame(0, index=dates, columns=["I", "R", "D", "C"], dtype="int64")
record_counts = pd.Series(0, index=dates, dtype="int64")
with pd.read_csv(SOURCE_FILE, usecols=columns, chunksize=CHUNK_SIZE,
                 na_values={"FECHA_DEF": ["9999-99-99"]}) as reader:
    for chunk in reader:
        admission = pd.to_datetime(chunk["FECHA_INGRESO"], format="%Y-%m-%d")
        records = chunk.loc[admission.between(START_DATE, END_DATE)].copy()
        if records.empty:
            continue
        counts = admission.loc[records.index].value_counts()
        record_counts.loc[counts.index] += counts
        records["FECHA_DEF"] = pd.to_datetime(records["FECHA_DEF"], format="%Y-%m-%d")
        _, chunk_obs = get_Y_obs(records, START_DATE, END_DATE, gamma_time=RECOVERY_DAYS)
        observed += chunk_obs[observed.columns]

Y = observed[["I", "R", "D"]].to_numpy(dtype=float)
assert record_counts.gt(0).all(), "Check source coverage on days without admission records."
assert (observed["C"] == observed[["I", "R", "D"]].sum(axis=1)).all()
assert observed[["C", "R", "D"]].diff().iloc[1:].ge(0).all().all()
print(f"{record_counts.sum():,} admission records across {len(dates):,} dates.")
print(observed.loc[[START_DATE, phases["End Date"].max(), END_DATE]].to_string())

19,965,086 admission records across 1,167 dates.
                I        R       D        C
Date                                       
2020-02-28      2        0       0        2
2023-03-31  30539  7192568  333755  7556862
2023-05-09  16932  7255491  334144  7606567


## Retain the forecast and analysis from the same ensemble

For each date after initialization, the forecast advances the previous day's analyzed members with the SIRD model and a Gaussian random walk on the log rates. The analysis assimilates that date's observation of $I,R,D$. The susceptible count is updated through population balance. On day 0, the first observation initializes the ensemble, so the forecast and correction are undefined.

In [4]:
rng = np.random.default_rng(SEED)
rate_step = partial(gaussian_rate_walk, log_step_std=LOG_STEP_STD)
members = initialize_sird_ensemble(
    Y[0], POPULATION, ENSEMBLE_SIZE, STATE_STD, INITIAL_RATES, INITIAL_LOG_STD, rng,
)

compartments = ["S", "I", "R", "D"]
forecast_mean = np.full((len(dates), 4), np.nan)
analysis_mean = np.empty((len(dates), 7))
rate_sd = np.empty((len(dates), 3))

for k in range(len(dates)):
    if k > 0:
        forecast = predict_sird_ensemble(members, POPULATION, day[k - 1], day[k], rate_step, rng)
        forecast_mean[k] = forecast[:, :4].mean(axis=0)
        members = analyze_sird_ensemble(forecast, Y[k], POPULATION, OBS_COV, rng)
    analysis_mean[k] = members.mean(axis=0)
    rate_sd[k] = members[:, 4:].std(axis=0, ddof=1)

forecast_counts = pd.DataFrame(forecast_mean, index=dates, columns=compartments)
filtered_counts = pd.DataFrame(analysis_mean[:, :4], index=dates, columns=compartments)
correction = filtered_counts - forecast_counts
np.testing.assert_allclose(filtered_counts.sum(axis=1), POPULATION, rtol=0, atol=1e-5)
np.testing.assert_allclose(correction.iloc[1:].sum(axis=1), 0, rtol=0, atol=1e-5)

## Export the estimated rates

Each row in `Datos/first_estimation_daily_parameters.csv` contains the date, day, period and the ensemble mean and sample standard deviation of each rate in day⁻¹. Day 0 reports the initialized rates. Subsequent rows report the rates after assimilation. The last column gives the reproduction ratio under full susceptibility, computed from the three mean rates.

In [5]:
parameters = calendar.copy()
for j, rate in enumerate(["beta", "gamma", "delta"]):
    parameters[f"{rate}_mean"] = analysis_mean[:, 4 + j]
    parameters[f"{rate}_sd"] = rate_sd[:, j]
parameters["R0"] = parameters["beta_mean"] / (parameters["gamma_mean"] + parameters["delta_mean"])
csv_path = DATA_DIR / "first_estimation_daily_parameters.csv"
parameters.to_csv(csv_path, index_label="Date", date_format="%Y-%m-%d")
print(f"Saved {len(parameters):,} daily estimates to {csv_path.relative_to(root)}")
phase_names = phases["Epidemiological Phase"]
phase_closings = parameters[parameters["Period"].isin(phase_names)].groupby("Period", sort=False).tail(1)
print(phase_closings[["Period", "beta_mean", "gamma_mean", "delta_mean", "R0"]].to_string())

Saved 1,167 daily estimates to Datos/first_estimation_daily_parameters.csv
                  Period  beta_mean  gamma_mean  delta_mean        R0
Date                                                                 
2020-03-22  Pre-lockdown   0.100980    0.011011    0.000954  8.439823
2020-09-30    First Wave   0.093483    0.054455    0.005445  1.560655
2021-04-30   Second Wave   0.069371    0.078668    0.004110  0.838046
2021-11-30    Third Wave   0.189115    0.168263    0.003518  1.100908
2022-03-31   Fourth Wave   0.079347    0.106199    0.001318  0.738000
2022-10-31    Fifth Wave   0.160785    0.266348    0.000836  0.601780
2023-03-31    Sixth Wave   0.070143    0.084563    0.000337  0.826196


## Compartment trajectories and daily corrections

Read each row from left to right. The left panel compares the daily SIRD forecast with the EnKF analysis and marks the input observations for $I,R,D$. The right panel shows the same day's analysis mean minus its forecast mean, in people. Values above zero indicate an upward correction. Each panel has its own vertical scale so the correction is visible even when the trajectories overlap.

Hover over either panel for the nearest date and its values. Drag horizontally between two dates to zoom all eight panels together. The zoom buttons change the time window around its center, and **Reset** restores the full phase. Open the notebook in a trusted Jupyter session to enable the embedded interaction.

In [6]:
PHASE_CHART = r'''
<!doctype html>
<html lang="en"><meta charset="utf-8"><style>
* {box-sizing:border-box} body {margin:8px;color:#20333f;background:white;font:13px system-ui,sans-serif}
h3 {margin:0 0 7px} header,.legend {display:flex;align-items:center;gap:12px;flex-wrap:wrap;margin-bottom:8px}
button {background:white;color:#20333f;border:1px solid #aab6bc;border-radius:4px;padding:3px 10px;cursor:pointer}
.grid {display:grid;grid-template-columns:minmax(0,1.2fr) minmax(0,1fr);gap:6px 12px} h4 {margin:2px 0}
canvas {width:100%;height:145px;display:block;touch-action:none;cursor:crosshair}
#tip {position:fixed;pointer-events:none;z-index:2;background:#fff;border:1px solid #8399a4;
border-radius:4px;padding:7px 10px;white-space:pre;box-shadow:0 2px 8px #0002;font-variant-numeric:tabular-nums}
</style>
<h3 id="phase"></h3>
<header><button id="closer">Zoom in</button><button id="wider">Zoom out</button>
<button id="reset">Reset</button><span id="range"></span></header>
<div class="legend"><span style="color:#ae4c00">┄ SIRD forecast</span>
<span style="color:#134e70">━ EnKF analysis</span><span>○ Observation</span>
<span style="color:#24776f">▮ Analysis − forecast</span></div>
<div class="grid"><h4>Compartment size (people)</h4><h4>Daily correction (people)</h4></div>
<div id="tip" role="status" hidden></div>
<script>
const data = __PHASE_DATA__;
const dates = data.dates, panels = [], grid = document.querySelector(".grid");
const tip = document.querySelector("#tip");
const fmt = new Intl.NumberFormat("en-US", {maximumFractionDigits: 4});
const last = dates.length - 1;
let lo = 0, hi = last, drag = null, hover = null;
document.querySelector("#phase").textContent = data.name;

function draw(panel) {
  const {canvas, item, isChange} = panel;
  const w = canvas.clientWidth, h = canvas.clientHeight;
  if (w < 1 || h < 1) return;
  const scale = window.devicePixelRatio || 1;
  if (canvas.width !== Math.round(w * scale) || canvas.height !== Math.round(h * scale)) {
    canvas.width = Math.round(w * scale);
    canvas.height = Math.round(h * scale);
  }
  const ctx = canvas.getContext("2d");
  ctx.setTransform(scale, 0, 0, scale, 0, 0);
  ctx.clearRect(0, 0, w, h);
  ctx.font = "11px system-ui";
  const left = Math.min(89, w * 0.30), right = w - 18, top = 22, bottom = h - 24;
  const step = (right - left) / (hi - lo + 1);
  const x = i => left + (i - lo + 0.5) * step;
  panel.indexAt = clientX => Math.max(lo, Math.min(hi,
    lo + Math.floor((clientX - canvas.getBoundingClientRect().left - left) / step)));
  const series = isChange ? [item.change] : [item.prior, item.filtered, item.obs];
  const values = series.flatMap(s => s.slice(lo, hi + 1)).filter(Number.isFinite);
  if (isChange) values.push(0);
  let min = Math.min(...values), max = Math.max(...values);
  const pad = (max - min) * 0.05 || 0.5;
  min -= pad;
  max += pad;
  const y = v => bottom - (v - min) / (max - min) * (bottom - top);
  ctx.fillStyle = "#20333f";
  ctx.fillText(item.label, left, 13);
  ctx.textAlign = "right";
  for (let j = 0; j < 4; j++) {
    const v = min + j / 3 * (max - min);
    ctx.fillText(v.toLocaleString("en-US", {maximumFractionDigits: max-min < 3 ? 2 : 0}), left - 5, y(v) + 4);
    ctx.fillStyle = "#e1e7ea";
    ctx.fillRect(left, y(v), right - left, 1);
    ctx.fillStyle = "#20333f";
  }
  ctx.textAlign = "left";
  ctx.fillText(dates[lo], left, h - 5);
  ctx.textAlign = "right";
  ctx.fillText(dates[hi], right, h - 5);
  ctx.textAlign = "left";
  if (isChange) {
    ctx.fillStyle = "#8b979c";
    ctx.fillRect(left, y(0), right - left, 1);
    ctx.fillStyle = "#24776f";
    for (let i = lo; i <= hi; i++) {
      if (item.change[i] !== null)
        ctx.fillRect(x(i) - step * 0.38, y(0), step * 0.76, y(item.change[i]) - y(0));
    }
  } else {
    for (const [s, color, dash] of [[item.filtered,"#134e70",[]],[item.prior,"#ae4c00",[5,3]]]) {
      ctx.beginPath();
      ctx.setLineDash(dash);
      ctx.strokeStyle = color;
      ctx.lineWidth = 1.6;
      let started = false;
      for (let i = lo; i <= hi; i++) {
        if (s[i] === null) { started = false; continue; }
        if (started) ctx.lineTo(x(i),y(s[i])); else ctx.moveTo(x(i),y(s[i]));
        started = true;
      }
      ctx.stroke();
    }
    ctx.setLineDash([]);
    ctx.strokeStyle = "#252525";
    ctx.lineWidth = 0.8;
    for (let i = lo; i <= hi; i++) {
      if (item.obs[i] === null) continue;
      ctx.beginPath();
      ctx.arc(x(i), y(item.obs[i]), 2, 0, 2*Math.PI);
      ctx.stroke();
    }
  }
  if (hover !== null) {
    ctx.fillStyle = "#43515c99";
    ctx.fillRect(x(hover), top, 1, bottom - top);
  }
  if (drag !== null && hover !== null) {
    ctx.fillStyle = "#2570a022";
    ctx.fillRect(x(drag), top, x(hover) - x(drag), bottom - top);
  }
}

function refresh() {
  document.querySelector("#range").textContent = dates[lo] + " – " + dates[hi];
  document.querySelector("#closer").disabled = hi - lo <= 1;
  document.querySelector("#wider").disabled = hi - lo === last;
  panels.forEach(draw);
}
function setWindow(a, b) {
  lo = a; hi = b; hover = drag = null; tip.hidden = true; refresh();
}
function zoom(factor) {
  const span = Math.min(last, Math.max(1, Math.round((hi-lo)*factor)));
  const a = Math.max(0, Math.min(last-span, Math.floor((lo+hi-span)/2)));
  setWindow(a, a+span);
}
document.querySelector("#closer").onclick = () => zoom(0.5);
document.querySelector("#wider").onclick = () => zoom(2);
document.querySelector("#reset").onclick = () => setWindow(0,last);

for (const item of data.items) {
  for (const isChange of [false, true]) {
    const canvas = document.createElement("canvas");
    canvas.setAttribute("aria-label", item.label + (isChange ? " daily correction" : " forecast and analysis"));
    grid.appendChild(canvas);
    const panel = {canvas, item, isChange};
    panels.push(panel);
    canvas.onpointermove = event => {
      hover = panel.indexAt(event.clientX);
      const entries = [["Forecast",item.prior],["Analysis",item.filtered],
                       ["Observation",item.obs],["Correction",item.change]];
      tip.textContent = [dates[hover] + " · " + item.label, ...entries
        .filter(([label]) => label !== "Observation" || item.key !== "S")
        .map(([label,s]) => label + ": " + (s[hover] === null ? "—" : fmt.format(s[hover])))].join("\n");
      tip.hidden = false;
      tip.style.left = Math.max(0, Math.min(event.clientX+14, innerWidth-tip.offsetWidth-4)) + "px";
      tip.style.top = Math.max(0, Math.min(event.clientY+12, innerHeight-tip.offsetHeight-4)) + "px";
      refresh();
    };
    canvas.onpointerdown = event => {
      drag = hover = panel.indexAt(event.clientX);
      canvas.setPointerCapture(event.pointerId);
    };
    canvas.onpointerup = event => {
      const end = panel.indexAt(event.clientX);
      if (drag !== null && end !== drag) setWindow(Math.min(drag,end), Math.max(drag,end));
      drag = null;
      refresh();
    };
    canvas.onpointercancel = () => { drag = hover = null; tip.hidden = true; refresh(); };
    canvas.onpointerleave = () => { if (drag === null) { hover = null; tip.hidden = true; refresh(); } };
  }
}
window.addEventListener("resize", refresh);
refresh();
</script></html>
'''

In [7]:
def plot_phase(name, start, end):
    window = slice(start, end)
    phase_dates = filtered_counts.loc[window].index
    items = []
    names = ["Susceptible (S)", "Active (I)", "Recovered (R)", "Deaths (D)"]
    for key, label in zip(compartments, names):
        series = pd.DataFrame({
            "prior": forecast_counts.loc[window, key],
            "filtered": filtered_counts.loc[window, key],
            "change": correction.loc[window, key],
        })
        series["obs"] = observed.loc[window, key] if key in ("I", "R", "D") else np.nan
        # JSON null denotes an unavailable observation or the day-zero forecast.
        values = {column: json.loads(series[column].to_json(orient="values", double_precision=15))
                  for column in series}
        items.append({"key": key, "label": label, **values})
    payload = {"name": name, "dates": phase_dates.strftime("%Y-%m-%d").tolist(), "items": items}
    encoded = json.dumps(payload, allow_nan=False).replace("<", r"\u003c")
    document = PHASE_CHART.replace("__PHASE_DATA__", encoded)
    frame = (f'<iframe title="{escape(name, quote=True)}" sandbox="allow-scripts" '
                 f'style="width:100%;height:770px;border:0" srcdoc="{escape(document, quote=True)}"></iframe>')
    display({"text/html": frame}, raw=True)


for name, start, end in phases[["Epidemiological Phase", "Start Date", "End Date"]].itertuples(index=False, name=None):
    plot_phase(name, start, end)

<iframe title="Pre-lockdown" sandbox="allow-scripts" style="width:100%;height:770px;border:0" srcdoc="
<!doctype html>
<html lang="en"><meta charset="utf-8"><style>
* {box-sizing:border-box} body {margin:8px;color:#20333f;background:white;font:13px system-ui,sans-serif}
h3 {margin:0 0 7px} header,.legend {display:flex;align-items:center;gap:12px;flex-wrap:wrap;margin-bottom:8px}
button {background:white;color:#20333f;border:1px solid #aab6bc;border-radius:4px;padding:3px 10px;cursor:pointer}
.grid {display:grid;grid-template-columns:minmax(0,1.2fr) minmax(0,1fr);gap:6px 12px} h4 {margin:2px 0}
canvas {width:100%;height:145px;display:block;touch-action:none;cursor:crosshair}
#tip {position:fixed;pointer-events:none;z-index:2;background:#fff;border:1px solid #8399a4;
border-radius:4px;padding:7px 10px;white-space:pre;box-shadow:0 2px 8px #0002;font-variant-numeric:tabular-nums}
</style>
<h3 id="phase"></h3>
<header><button id="closer">Zoom in</button><button id="wider">Zoom out</button>
<button id="reset">Reset</button><span id="range"></span></header>
<div class="legend"><span style="color:#ae4c00">┄ SIRD forecast</span>
<span style="color:#134e70">━ EnKF analysis</span><span>○ Observation</span>
<span style="color:#24776f">▮ Analysis − forecast</span></div>
<div class="grid"><h4>Compartment size (people)</h4><h4>Daily correction (people)</h4></div>
<div id="tip" role="status" hidden></div>
<script>
const data = {"name": "Pre-lockdown", "dates": ["2020-02-28", "2020-02-29", "2020-03-01", "2020-03-02", "2020-03-03", "2020-03-04", "2020-03-05", "2020-03-06", "2020-03-07", "2020-03-08", "2020-03-09", "2020-03-10", "2020-03-11", "2020-03-12", "2020-03-13", "2020-03-14", "2020-03-15", "2020-03-16", "2020-03-17", "2020-03-18", "2020-03-19", "2020-03-20", "2020-03-21", "2020-03-22"], "items": [{"key": "S", "label": "Susceptible (S)", "prior": [null, 126014021.0529356, 126014019.71936293, 126014017.49751973, 126014012.58601992, 126013999.06546824, 126013977.9005884, 126013955.41812646, 126013946.22693916, 126013936.32737115, 126013928.43318117, 126013924.72860077, 126013909.63314211, 126013869.18821378, 126013810.19308417, 126013712.52411917, 126013645.80172372, 126013617.32332206, 126013535.66929266, 126013410.23017845, 126013275.84211417, 126013084.26038763, 126012944.49010122, 126012891.38559222], "filtered": [126014021.96840915, 126014021.05589983, 126014019.6224806, 126014016.40330872, 126014007.79832396, 126013994.8650845, 126013980.01395693, 126013971.75829914, 126013961.81401086, 126013952.75778902, 126013945.75910139, 126013932.3345572, 126013902.35832651, 126013860.77930292, 126013795.48279476, 126013734.95135888, 126013693.83169222, 126013624.78045669, 126013526.63354251, 126013412.858695, 126013262.98082404, 126013122.52576551, 126013029.86247888, 126012955.95101073], "change": [null, 0.002964228391647, -0.096882328391075, -1.094211012125015, -4.787695959210396, -4.200383737683296, 2.113368541002274, 16.340172678232193, 15.58707170188427, 16.430417865514755, 17.32592022418976, 7.605956435203552, -7.274815604090691, -8.408910855650902, -14.710289403796196, 22.42723971605301, 48.02996850013733, 7.457134634256363, -9.035750150680542, 2.628516554832458, -12.861290127038956, 38.26537787914276, 85.37237766385078, 64.5654185116291], "obs": [null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null]}, {"key": "I", "label": "Active (I)", "prior": [null, 2.75738439968643, 3.834462376113987, 5.679589413966213, 9.862529739640681, 21.605102479539166, 40.45076590854405, 61.49036475261124, 71.4661416153019, 81.2735204888196, 89.14256892060135, 92.55540915233306, 107.08022515010248, 146.86296177065407, 204.88568445499033, 301.42598051646667, 367.1564144782742, 395.0384241869796, 473.7326949464402, 593.6091922574354, 718.4136618866912, 895.8616541167497, 1022.6062875174978, 1065.545483826962], "filtered": [2.031590790859513, 2.754780296031187, 3.92703659304

<iframe title="First Wave" sandbox="allow-scripts" style="width:100%;height:770px;border:0" srcdoc="
<!doctype html>
<html lang="en"><meta charset="utf-8"><style>
* {box-sizing:border-box} body {margin:8px;color:#20333f;background:white;font:13px system-ui,sans-serif}
h3 {margin:0 0 7px} header,.legend {display:flex;align-items:center;gap:12px;flex-wrap:wrap;margin-bottom:8px}
button {background:white;color:#20333f;border:1px solid #aab6bc;border-radius:4px;padding:3px 10px;cursor:pointer}
.grid {display:grid;grid-template-columns:minmax(0,1.2fr) minmax(0,1fr);gap:6px 12px} h4 {margin:2px 0}
canvas {width:100%;height:145px;display:block;touch-action:none;cursor:crosshair}
#tip {position:fixed;pointer-events:none;z-index:2;background:#fff;border:1px solid #8399a4;
border-radius:4px;padding:7px 10px;white-space:pre;box-shadow:0 2px 8px #0002;font-variant-numeric:tabular-nums}
</style>
<h3 id="phase"></h3>
<header><button id="closer">Zoom in</button><button id="wider">Zoom out</button>
<button id="reset">Reset</button><span id="range"></span></header>
<div class="legend"><span style="color:#ae4c00">┄ SIRD forecast</span>
<span style="color:#134e70">━ EnKF analysis</span><span>○ Observation</span>
<span style="color:#24776f">▮ Analysis − forecast</span></div>
<div class="grid"><h4>Compartment size (people)</h4><h4>Daily correction (people)</h4></div>
<div id="tip" role="status" hidden></div>
<script>
const data = {"name": "First Wave", "dates": ["2020-03-23", "2020-03-24", "2020-03-25", "2020-03-26", "2020-03-27", "2020-03-28", "2020-03-29", "2020-03-30", "2020-03-31", "2020-04-01", "2020-04-02", "2020-04-03", "2020-04-04", "2020-04-05", "2020-04-06", "2020-04-07", "2020-04-08", "2020-04-09", "2020-04-10", "2020-04-11", "2020-04-12", "2020-04-13", "2020-04-14", "2020-04-15", "2020-04-16", "2020-04-17", "2020-04-18", "2020-04-19", "2020-04-20", "2020-04-21", "2020-04-22", "2020-04-23", "2020-04-24", "2020-04-25", "2020-04-26", "2020-04-27", "2020-04-28", "2020-04-29", "2020-04-30", "2020-05-01", "2020-05-02", "2020-05-03", "2020-05-04", "2020-05-05", "2020-05-06", "2020-05-07", "2020-05-08", "2020-05-09", "2020-05-10", "2020-05-11", "2020-05-12", "2020-05-13", "2020-05-14", "2020-05-15", "2020-05-16", "2020-05-17", "2020-05-18", "2020-05-19", "2020-05-20", "2020-05-21", "2020-05-22", "2020-05-23", "2020-05-24", "2020-05-25", "2020-05-26", "2020-05-27", "2020-05-28", "2020-05-29", "2020-05-30", "2020-05-31", "2020-06-01", "2020-06-02", "2020-06-03", "2020-06-04", "2020-06-05", "2020-06-06", "2020-06-07", "2020-06-08", "2020-06-09", "2020-06-10", "2020-06-11", "2020-06-12", "2020-06-13", "2020-06-14", "2020-06-15", "2020-06-16", "2020-06-17", "2020-06-18", "2020-06-19", "2020-06-20", "2020-06-21", "2020-06-22", "2020-06-23", "2020-06-24", "2020-06-25", "2020-06-26", "2020-06-27", "2020-06-28", "2020-06-29", "2020-06-30", "2020-07-01", "2020-07-02", "2020-07-03", "2020-07-04", "2020-07-05", "2020-07-06", "2020-07-07", "2020-07-08", "2020-07-09", "2020-07-10", "2020-07-11", "2020-07-12", "2020-07-13", "2020-07-14", "2020-07-15", "2020-07-16", "2020-07-17", "2020-07-18", "2020-07-19", "2020-07-20", "2020-07-21", "2020-07-22", "2020-07-23", "2020-07-24", "2020-07-25", "2020-07-26", "2020-07-27", "2020-07-28", "2020-07-29", "2020-07-30", "2020-07-31", "2020-08-01", "2020-08-02", "2020-08-03", "2020-08-04", "2020-08-05", "2020-08-06", "2020-08-07", "2020-08-08", "2020-08-09", "2020-08-10", "2020-08-11", "2020-08-12", "2020-08-13", "2020-08-14", "2020-08-15", "2020-08-16", "2020-08-17", "2020-08-18", "2020-08-19", "2020-08-20", "2020-08-21", "2020-08-22", "2020-08-23", "2020-08-24", "2020-08-25", "2020-08-26", "2020-08-27", "2020-08-28", "2020-08-29", "2020-08-30", "2020-08-31", "2020-09-01", "2020-09-02", "2020-09-03", "2020-09-04", "2020-09-05", "2020-09-06", "2020-09-07", "2020-09-08", "2020-09-09", "2020-09-10", "2020-09-11", "2020-09-12", "2020-09-13", "2020-09-14", "2020-09-15", "2020-09-16", "2020-09-17", "2020-09-18", "2020-09-19", 

<iframe title="Second Wave" sandbox="allow-scripts" style="width:100%;height:770px;border:0" srcdoc="
<!doctype html>
<html lang="en"><meta charset="utf-8"><style>
* {box-sizing:border-box} body {margin:8px;color:#20333f;background:white;font:13px system-ui,sans-serif}
h3 {margin:0 0 7px} header,.legend {display:flex;align-items:center;gap:12px;flex-wrap:wrap;margin-bottom:8px}
button {background:white;color:#20333f;border:1px solid #aab6bc;border-radius:4px;padding:3px 10px;cursor:pointer}
.grid {display:grid;grid-template-columns:minmax(0,1.2fr) minmax(0,1fr);gap:6px 12px} h4 {margin:2px 0}
canvas {width:100%;height:145px;display:block;touch-action:none;cursor:crosshair}
#tip {position:fixed;pointer-events:none;z-index:2;background:#fff;border:1px solid #8399a4;
border-radius:4px;padding:7px 10px;white-space:pre;box-shadow:0 2px 8px #0002;font-variant-numeric:tabular-nums}
</style>
<h3 id="phase"></h3>
<header><button id="closer">Zoom in</button><button id="wider">Zoom out</button>
<button id="reset">Reset</button><span id="range"></span></header>
<div class="legend"><span style="color:#ae4c00">┄ SIRD forecast</span>
<span style="color:#134e70">━ EnKF analysis</span><span>○ Observation</span>
<span style="color:#24776f">▮ Analysis − forecast</span></div>
<div class="grid"><h4>Compartment size (people)</h4><h4>Daily correction (people)</h4></div>
<div id="tip" role="status" hidden></div>
<script>
const data = {"name": "Second Wave", "dates": ["2020-10-01", "2020-10-02", "2020-10-03", "2020-10-04", "2020-10-05", "2020-10-06", "2020-10-07", "2020-10-08", "2020-10-09", "2020-10-10", "2020-10-11", "2020-10-12", "2020-10-13", "2020-10-14", "2020-10-15", "2020-10-16", "2020-10-17", "2020-10-18", "2020-10-19", "2020-10-20", "2020-10-21", "2020-10-22", "2020-10-23", "2020-10-24", "2020-10-25", "2020-10-26", "2020-10-27", "2020-10-28", "2020-10-29", "2020-10-30", "2020-10-31", "2020-11-01", "2020-11-02", "2020-11-03", "2020-11-04", "2020-11-05", "2020-11-06", "2020-11-07", "2020-11-08", "2020-11-09", "2020-11-10", "2020-11-11", "2020-11-12", "2020-11-13", "2020-11-14", "2020-11-15", "2020-11-16", "2020-11-17", "2020-11-18", "2020-11-19", "2020-11-20", "2020-11-21", "2020-11-22", "2020-11-23", "2020-11-24", "2020-11-25", "2020-11-26", "2020-11-27", "2020-11-28", "2020-11-29", "2020-11-30", "2020-12-01", "2020-12-02", "2020-12-03", "2020-12-04", "2020-12-05", "2020-12-06", "2020-12-07", "2020-12-08", "2020-12-09", "2020-12-10", "2020-12-11", "2020-12-12", "2020-12-13", "2020-12-14", "2020-12-15", "2020-12-16", "2020-12-17", "2020-12-18", "2020-12-19", "2020-12-20", "2020-12-21", "2020-12-22", "2020-12-23", "2020-12-24", "2020-12-25", "2020-12-26", "2020-12-27", "2020-12-28", "2020-12-29", "2020-12-30", "2020-12-31", "2021-01-01", "2021-01-02", "2021-01-03", "2021-01-04", "2021-01-05", "2021-01-06", "2021-01-07", "2021-01-08", "2021-01-09", "2021-01-10", "2021-01-11", "2021-01-12", "2021-01-13", "2021-01-14", "2021-01-15", "2021-01-16", "2021-01-17", "2021-01-18", "2021-01-19", "2021-01-20", "2021-01-21", "2021-01-22", "2021-01-23", "2021-01-24", "2021-01-25", "2021-01-26", "2021-01-27", "2021-01-28", "2021-01-29", "2021-01-30", "2021-01-31", "2021-02-01", "2021-02-02", "2021-02-03", "2021-02-04", "2021-02-05", "2021-02-06", "2021-02-07", "2021-02-08", "2021-02-09", "2021-02-10", "2021-02-11", "2021-02-12", "2021-02-13", "2021-02-14", "2021-02-15", "2021-02-16", "2021-02-17", "2021-02-18", "2021-02-19", "2021-02-20", "2021-02-21", "2021-02-22", "2021-02-23", "2021-02-24", "2021-02-25", "2021-02-26", "2021-02-27", "2021-02-28", "2021-03-01", "2021-03-02", "2021-03-03", "2021-03-04", "2021-03-05", "2021-03-06", "2021-03-07", "2021-03-08", "2021-03-09", "2021-03-10", "2021-03-11", "2021-03-12", "2021-03-13", "2021-03-14", "2021-03-15", "2021-03-16", "2021-03-17", "2021-03-18", "2021-03-19", "2021-03-20", "2021-03-21", "2021-03-22", "2021-03-23", "2021-03-24", "2021-03-25", "2021-03-26", "2021-03-27", "2021-03-28", "2021-03-29", "2021-03-30"

<iframe title="Third Wave" sandbox="allow-scripts" style="width:100%;height:770px;border:0" srcdoc="
<!doctype html>
<html lang="en"><meta charset="utf-8"><style>
* {box-sizing:border-box} body {margin:8px;color:#20333f;background:white;font:13px system-ui,sans-serif}
h3 {margin:0 0 7px} header,.legend {display:flex;align-items:center;gap:12px;flex-wrap:wrap;margin-bottom:8px}
button {background:white;color:#20333f;border:1px solid #aab6bc;border-radius:4px;padding:3px 10px;cursor:pointer}
.grid {display:grid;grid-template-columns:minmax(0,1.2fr) minmax(0,1fr);gap:6px 12px} h4 {margin:2px 0}
canvas {width:100%;height:145px;display:block;touch-action:none;cursor:crosshair}
#tip {position:fixed;pointer-events:none;z-index:2;background:#fff;border:1px solid #8399a4;
border-radius:4px;padding:7px 10px;white-space:pre;box-shadow:0 2px 8px #0002;font-variant-numeric:tabular-nums}
</style>
<h3 id="phase"></h3>
<header><button id="closer">Zoom in</button><button id="wider">Zoom out</button>
<button id="reset">Reset</button><span id="range"></span></header>
<div class="legend"><span style="color:#ae4c00">┄ SIRD forecast</span>
<span style="color:#134e70">━ EnKF analysis</span><span>○ Observation</span>
<span style="color:#24776f">▮ Analysis − forecast</span></div>
<div class="grid"><h4>Compartment size (people)</h4><h4>Daily correction (people)</h4></div>
<div id="tip" role="status" hidden></div>
<script>
const data = {"name": "Third Wave", "dates": ["2021-05-01", "2021-05-02", "2021-05-03", "2021-05-04", "2021-05-05", "2021-05-06", "2021-05-07", "2021-05-08", "2021-05-09", "2021-05-10", "2021-05-11", "2021-05-12", "2021-05-13", "2021-05-14", "2021-05-15", "2021-05-16", "2021-05-17", "2021-05-18", "2021-05-19", "2021-05-20", "2021-05-21", "2021-05-22", "2021-05-23", "2021-05-24", "2021-05-25", "2021-05-26", "2021-05-27", "2021-05-28", "2021-05-29", "2021-05-30", "2021-05-31", "2021-06-01", "2021-06-02", "2021-06-03", "2021-06-04", "2021-06-05", "2021-06-06", "2021-06-07", "2021-06-08", "2021-06-09", "2021-06-10", "2021-06-11", "2021-06-12", "2021-06-13", "2021-06-14", "2021-06-15", "2021-06-16", "2021-06-17", "2021-06-18", "2021-06-19", "2021-06-20", "2021-06-21", "2021-06-22", "2021-06-23", "2021-06-24", "2021-06-25", "2021-06-26", "2021-06-27", "2021-06-28", "2021-06-29", "2021-06-30", "2021-07-01", "2021-07-02", "2021-07-03", "2021-07-04", "2021-07-05", "2021-07-06", "2021-07-07", "2021-07-08", "2021-07-09", "2021-07-10", "2021-07-11", "2021-07-12", "2021-07-13", "2021-07-14", "2021-07-15", "2021-07-16", "2021-07-17", "2021-07-18", "2021-07-19", "2021-07-20", "2021-07-21", "2021-07-22", "2021-07-23", "2021-07-24", "2021-07-25", "2021-07-26", "2021-07-27", "2021-07-28", "2021-07-29", "2021-07-30", "2021-07-31", "2021-08-01", "2021-08-02", "2021-08-03", "2021-08-04", "2021-08-05", "2021-08-06", "2021-08-07", "2021-08-08", "2021-08-09", "2021-08-10", "2021-08-11", "2021-08-12", "2021-08-13", "2021-08-14", "2021-08-15", "2021-08-16", "2021-08-17", "2021-08-18", "2021-08-19", "2021-08-20", "2021-08-21", "2021-08-22", "2021-08-23", "2021-08-24", "2021-08-25", "2021-08-26", "2021-08-27", "2021-08-28", "2021-08-29", "2021-08-30", "2021-08-31", "2021-09-01", "2021-09-02", "2021-09-03", "2021-09-04", "2021-09-05", "2021-09-06", "2021-09-07", "2021-09-08", "2021-09-09", "2021-09-10", "2021-09-11", "2021-09-12", "2021-09-13", "2021-09-14", "2021-09-15", "2021-09-16", "2021-09-17", "2021-09-18", "2021-09-19", "2021-09-20", "2021-09-21", "2021-09-22", "2021-09-23", "2021-09-24", "2021-09-25", "2021-09-26", "2021-09-27", "2021-09-28", "2021-09-29", "2021-09-30", "2021-10-01", "2021-10-02", "2021-10-03", "2021-10-04", "2021-10-05", "2021-10-06", "2021-10-07", "2021-10-08", "2021-10-09", "2021-10-10", "2021-10-11", "2021-10-12", "2021-10-13", "2021-10-14", "2021-10-15", "2021-10-16", "2021-10-17", "2021-10-18", "2021-10-19", "2021-10-20", "2021-10-21", "2021-10-22", "2021-10-23", "2021-10-24", "2021-10-25", "2021-10-26", "2021-10-27", "2021-10-28", 

<iframe title="Fourth Wave" sandbox="allow-scripts" style="width:100%;height:770px;border:0" srcdoc="
<!doctype html>
<html lang="en"><meta charset="utf-8"><style>
* {box-sizing:border-box} body {margin:8px;color:#20333f;background:white;font:13px system-ui,sans-serif}
h3 {margin:0 0 7px} header,.legend {display:flex;align-items:center;gap:12px;flex-wrap:wrap;margin-bottom:8px}
button {background:white;color:#20333f;border:1px solid #aab6bc;border-radius:4px;padding:3px 10px;cursor:pointer}
.grid {display:grid;grid-template-columns:minmax(0,1.2fr) minmax(0,1fr);gap:6px 12px} h4 {margin:2px 0}
canvas {width:100%;height:145px;display:block;touch-action:none;cursor:crosshair}
#tip {position:fixed;pointer-events:none;z-index:2;background:#fff;border:1px solid #8399a4;
border-radius:4px;padding:7px 10px;white-space:pre;box-shadow:0 2px 8px #0002;font-variant-numeric:tabular-nums}
</style>
<h3 id="phase"></h3>
<header><button id="closer">Zoom in</button><button id="wider">Zoom out</button>
<button id="reset">Reset</button><span id="range"></span></header>
<div class="legend"><span style="color:#ae4c00">┄ SIRD forecast</span>
<span style="color:#134e70">━ EnKF analysis</span><span>○ Observation</span>
<span style="color:#24776f">▮ Analysis − forecast</span></div>
<div class="grid"><h4>Compartment size (people)</h4><h4>Daily correction (people)</h4></div>
<div id="tip" role="status" hidden></div>
<script>
const data = {"name": "Fourth Wave", "dates": ["2021-12-01", "2021-12-02", "2021-12-03", "2021-12-04", "2021-12-05", "2021-12-06", "2021-12-07", "2021-12-08", "2021-12-09", "2021-12-10", "2021-12-11", "2021-12-12", "2021-12-13", "2021-12-14", "2021-12-15", "2021-12-16", "2021-12-17", "2021-12-18", "2021-12-19", "2021-12-20", "2021-12-21", "2021-12-22", "2021-12-23", "2021-12-24", "2021-12-25", "2021-12-26", "2021-12-27", "2021-12-28", "2021-12-29", "2021-12-30", "2021-12-31", "2022-01-01", "2022-01-02", "2022-01-03", "2022-01-04", "2022-01-05", "2022-01-06", "2022-01-07", "2022-01-08", "2022-01-09", "2022-01-10", "2022-01-11", "2022-01-12", "2022-01-13", "2022-01-14", "2022-01-15", "2022-01-16", "2022-01-17", "2022-01-18", "2022-01-19", "2022-01-20", "2022-01-21", "2022-01-22", "2022-01-23", "2022-01-24", "2022-01-25", "2022-01-26", "2022-01-27", "2022-01-28", "2022-01-29", "2022-01-30", "2022-01-31", "2022-02-01", "2022-02-02", "2022-02-03", "2022-02-04", "2022-02-05", "2022-02-06", "2022-02-07", "2022-02-08", "2022-02-09", "2022-02-10", "2022-02-11", "2022-02-12", "2022-02-13", "2022-02-14", "2022-02-15", "2022-02-16", "2022-02-17", "2022-02-18", "2022-02-19", "2022-02-20", "2022-02-21", "2022-02-22", "2022-02-23", "2022-02-24", "2022-02-25", "2022-02-26", "2022-02-27", "2022-02-28", "2022-03-01", "2022-03-02", "2022-03-03", "2022-03-04", "2022-03-05", "2022-03-06", "2022-03-07", "2022-03-08", "2022-03-09", "2022-03-10", "2022-03-11", "2022-03-12", "2022-03-13", "2022-03-14", "2022-03-15", "2022-03-16", "2022-03-17", "2022-03-18", "2022-03-19", "2022-03-20", "2022-03-21", "2022-03-22", "2022-03-23", "2022-03-24", "2022-03-25", "2022-03-26", "2022-03-27", "2022-03-28", "2022-03-29", "2022-03-30", "2022-03-31"], "items": [{"key": "S", "label": "Susceptible (S)", "prior": [122068741.61273213, 122068643.47965094, 122066246.76938657, 122063886.8552167, 122064105.90304399, 122063736.60072418, 122047581.3782603, 122051736.7306441, 122051267.61593494, 122049112.21712692, 122046420.0027922, 122046831.58326133, 122046354.38285577, 122035552.90840307, 122037005.4341423, 122035358.53352956, 122032844.50807476, 122030532.92747264, 122030394.42548458, 122029851.845498, 122015215.46184702, 122017404.12757508, 122015076.78428784, 122011144.08228704, 122008585.2918545, 122009178.4789491, 122006841.66554472, 121939723.07704239, 121957441.99933739, 121958143.07524443, 121949403.61140028, 121941269.54682219, 121944383.41810584, 121939164.34057507, 121741528.9409771, 121783605.46588886, 121775855.95466387, 121747122.7932271, 121702685.85372564, 1217025

<iframe title="Fifth Wave" sandbox="allow-scripts" style="width:100%;height:770px;border:0" srcdoc="
<!doctype html>
<html lang="en"><meta charset="utf-8"><style>
* {box-sizing:border-box} body {margin:8px;color:#20333f;background:white;font:13px system-ui,sans-serif}
h3 {margin:0 0 7px} header,.legend {display:flex;align-items:center;gap:12px;flex-wrap:wrap;margin-bottom:8px}
button {background:white;color:#20333f;border:1px solid #aab6bc;border-radius:4px;padding:3px 10px;cursor:pointer}
.grid {display:grid;grid-template-columns:minmax(0,1.2fr) minmax(0,1fr);gap:6px 12px} h4 {margin:2px 0}
canvas {width:100%;height:145px;display:block;touch-action:none;cursor:crosshair}
#tip {position:fixed;pointer-events:none;z-index:2;background:#fff;border:1px solid #8399a4;
border-radius:4px;padding:7px 10px;white-space:pre;box-shadow:0 2px 8px #0002;font-variant-numeric:tabular-nums}
</style>
<h3 id="phase"></h3>
<header><button id="closer">Zoom in</button><button id="wider">Zoom out</button>
<button id="reset">Reset</button><span id="range"></span></header>
<div class="legend"><span style="color:#ae4c00">┄ SIRD forecast</span>
<span style="color:#134e70">━ EnKF analysis</span><span>○ Observation</span>
<span style="color:#24776f">▮ Analysis − forecast</span></div>
<div class="grid"><h4>Compartment size (people)</h4><h4>Daily correction (people)</h4></div>
<div id="tip" role="status" hidden></div>
<script>
const data = {"name": "Fifth Wave", "dates": ["2022-05-01", "2022-05-02", "2022-05-03", "2022-05-04", "2022-05-05", "2022-05-06", "2022-05-07", "2022-05-08", "2022-05-09", "2022-05-10", "2022-05-11", "2022-05-12", "2022-05-13", "2022-05-14", "2022-05-15", "2022-05-16", "2022-05-17", "2022-05-18", "2022-05-19", "2022-05-20", "2022-05-21", "2022-05-22", "2022-05-23", "2022-05-24", "2022-05-25", "2022-05-26", "2022-05-27", "2022-05-28", "2022-05-29", "2022-05-30", "2022-05-31", "2022-06-01", "2022-06-02", "2022-06-03", "2022-06-04", "2022-06-05", "2022-06-06", "2022-06-07", "2022-06-08", "2022-06-09", "2022-06-10", "2022-06-11", "2022-06-12", "2022-06-13", "2022-06-14", "2022-06-15", "2022-06-16", "2022-06-17", "2022-06-18", "2022-06-19", "2022-06-20", "2022-06-21", "2022-06-22", "2022-06-23", "2022-06-24", "2022-06-25", "2022-06-26", "2022-06-27", "2022-06-28", "2022-06-29", "2022-06-30", "2022-07-01", "2022-07-02", "2022-07-03", "2022-07-04", "2022-07-05", "2022-07-06", "2022-07-07", "2022-07-08", "2022-07-09", "2022-07-10", "2022-07-11", "2022-07-12", "2022-07-13", "2022-07-14", "2022-07-15", "2022-07-16", "2022-07-17", "2022-07-18", "2022-07-19", "2022-07-20", "2022-07-21", "2022-07-22", "2022-07-23", "2022-07-24", "2022-07-25", "2022-07-26", "2022-07-27", "2022-07-28", "2022-07-29", "2022-07-30", "2022-07-31", "2022-08-01", "2022-08-02", "2022-08-03", "2022-08-04", "2022-08-05", "2022-08-06", "2022-08-07", "2022-08-08", "2022-08-09", "2022-08-10", "2022-08-11", "2022-08-12", "2022-08-13", "2022-08-14", "2022-08-15", "2022-08-16", "2022-08-17", "2022-08-18", "2022-08-19", "2022-08-20", "2022-08-21", "2022-08-22", "2022-08-23", "2022-08-24", "2022-08-25", "2022-08-26", "2022-08-27", "2022-08-28", "2022-08-29", "2022-08-30", "2022-08-31", "2022-09-01", "2022-09-02", "2022-09-03", "2022-09-04", "2022-09-05", "2022-09-06", "2022-09-07", "2022-09-08", "2022-09-09", "2022-09-10", "2022-09-11", "2022-09-12", "2022-09-13", "2022-09-14", "2022-09-15", "2022-09-16", "2022-09-17", "2022-09-18", "2022-09-19", "2022-09-20", "2022-09-21", "2022-09-22", "2022-09-23", "2022-09-24", "2022-09-25", "2022-09-26", "2022-09-27", "2022-09-28", "2022-09-29", "2022-09-30", "2022-10-01", "2022-10-02", "2022-10-03", "2022-10-04", "2022-10-05", "2022-10-06", "2022-10-07", "2022-10-08", "2022-10-09", "2022-10-10", "2022-10-11", "2022-10-12", "2022-10-13", "2022-10-14", "2022-10-15", "2022-10-16", "2022-10-17", "2022-10-18", "2022-10-19", "2022-10-20", "2022-10-21", "2022-10-22", "2022-10-23", "2022-10-24", "2022-10-25", "2022-10-26", "2022-10-27", "2022-10-28", 

<iframe title="Sixth Wave" sandbox="allow-scripts" style="width:100%;height:770px;border:0" srcdoc="
<!doctype html>
<html lang="en"><meta charset="utf-8"><style>
* {box-sizing:border-box} body {margin:8px;color:#20333f;background:white;font:13px system-ui,sans-serif}
h3 {margin:0 0 7px} header,.legend {display:flex;align-items:center;gap:12px;flex-wrap:wrap;margin-bottom:8px}
button {background:white;color:#20333f;border:1px solid #aab6bc;border-radius:4px;padding:3px 10px;cursor:pointer}
.grid {display:grid;grid-template-columns:minmax(0,1.2fr) minmax(0,1fr);gap:6px 12px} h4 {margin:2px 0}
canvas {width:100%;height:145px;display:block;touch-action:none;cursor:crosshair}
#tip {position:fixed;pointer-events:none;z-index:2;background:#fff;border:1px solid #8399a4;
border-radius:4px;padding:7px 10px;white-space:pre;box-shadow:0 2px 8px #0002;font-variant-numeric:tabular-nums}
</style>
<h3 id="phase"></h3>
<header><button id="closer">Zoom in</button><button id="wider">Zoom out</button>
<button id="reset">Reset</button><span id="range"></span></header>
<div class="legend"><span style="color:#ae4c00">┄ SIRD forecast</span>
<span style="color:#134e70">━ EnKF analysis</span><span>○ Observation</span>
<span style="color:#24776f">▮ Analysis − forecast</span></div>
<div class="grid"><h4>Compartment size (people)</h4><h4>Daily correction (people)</h4></div>
<div id="tip" role="status" hidden></div>
<script>
const data = {"name": "Sixth Wave", "dates": ["2022-11-01", "2022-11-02", "2022-11-03", "2022-11-04", "2022-11-05", "2022-11-06", "2022-11-07", "2022-11-08", "2022-11-09", "2022-11-10", "2022-11-11", "2022-11-12", "2022-11-13", "2022-11-14", "2022-11-15", "2022-11-16", "2022-11-17", "2022-11-18", "2022-11-19", "2022-11-20", "2022-11-21", "2022-11-22", "2022-11-23", "2022-11-24", "2022-11-25", "2022-11-26", "2022-11-27", "2022-11-28", "2022-11-29", "2022-11-30", "2022-12-01", "2022-12-02", "2022-12-03", "2022-12-04", "2022-12-05", "2022-12-06", "2022-12-07", "2022-12-08", "2022-12-09", "2022-12-10", "2022-12-11", "2022-12-12", "2022-12-13", "2022-12-14", "2022-12-15", "2022-12-16", "2022-12-17", "2022-12-18", "2022-12-19", "2022-12-20", "2022-12-21", "2022-12-22", "2022-12-23", "2022-12-24", "2022-12-25", "2022-12-26", "2022-12-27", "2022-12-28", "2022-12-29", "2022-12-30", "2022-12-31", "2023-01-01", "2023-01-02", "2023-01-03", "2023-01-04", "2023-01-05", "2023-01-06", "2023-01-07", "2023-01-08", "2023-01-09", "2023-01-10", "2023-01-11", "2023-01-12", "2023-01-13", "2023-01-14", "2023-01-15", "2023-01-16", "2023-01-17", "2023-01-18", "2023-01-19", "2023-01-20", "2023-01-21", "2023-01-22", "2023-01-23", "2023-01-24", "2023-01-25", "2023-01-26", "2023-01-27", "2023-01-28", "2023-01-29", "2023-01-30", "2023-01-31", "2023-02-01", "2023-02-02", "2023-02-03", "2023-02-04", "2023-02-05", "2023-02-06", "2023-02-07", "2023-02-08", "2023-02-09", "2023-02-10", "2023-02-11", "2023-02-12", "2023-02-13", "2023-02-14", "2023-02-15", "2023-02-16", "2023-02-17", "2023-02-18", "2023-02-19", "2023-02-20", "2023-02-21", "2023-02-22", "2023-02-23", "2023-02-24", "2023-02-25", "2023-02-26", "2023-02-27", "2023-02-28", "2023-03-01", "2023-03-02", "2023-03-03", "2023-03-04", "2023-03-05", "2023-03-06", "2023-03-07", "2023-03-08", "2023-03-09", "2023-03-10", "2023-03-11", "2023-03-12", "2023-03-13", "2023-03-14", "2023-03-15", "2023-03-16", "2023-03-17", "2023-03-18", "2023-03-19", "2023-03-20", "2023-03-21", "2023-03-22", "2023-03-23", "2023-03-24", "2023-03-25", "2023-03-26", "2023-03-27", "2023-03-28", "2023-03-29", "2023-03-30", "2023-03-31"], "items": [{"key": "S", "label": "Susceptible (S)", "prior": [118879876.05087507, 118879524.20077741, 118879291.49790835, 118878397.20740359, 118877970.85893898, 118878008.68784568, 118877908.32403827, 118875435.09056747, 118875325.82941732, 118874789.62915166, 118873989.15354387, 118873404.43612063, 118873488.65151179, 118873327.99280028, 118869865.52039953, 118869862.39092009, 118869208.02498427, 118868423.98665003